In [6]:
import numpy as np
import pandas as pd

In [7]:
movies= pd.read_csv("/content/tmdb_5000_movies.csv")
credits= pd.read_csv("/content/tmdb_5000_credits.csv")

In [8]:
movies= movies.merge(credits,on='title') # Both merged and saved as movies

In [9]:
# genres,id,keywords,title,overview,cast,crew
movies= movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [10]:
# movies.isnull().sum()
movies.dropna(inplace=True)

In [11]:
movies.duplicated().sum() #Checking duplicate rows

np.int64(0)

In [12]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [13]:
import ast
# Getting all genres/keywords of a movie together
def convert(obj):
  L=[]
  for i in ast.literal_eval(obj):   # Converting string to integer
    L.append(i['name']);
  return L

In [14]:
movies['genres']=movies['genres'].apply(convert)

In [15]:
movies['keywords']=movies['keywords'].apply(convert)

In [16]:
movies['cast'][0] # Before

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [17]:
# Getting names of only first three actors of a movie
def convert3(obj):
  L=[]
  counter=0;
  for i in ast.literal_eval(obj):
    if counter!=3:
       L.append(i['name']);
       counter+=1;
    else:
      break;
  return L

In [18]:
movies['cast']=movies['cast'].apply(convert3)

In [19]:
movies['cast'][0] # After

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']

In [20]:
# Keeping only director name in the crew
def fetch_director(obj):
  L=[]
  for i in ast.literal_eval(obj):   # Converting string to integer
    if i['job']=='Director':
      L.append(i['name']);
  return L

In [21]:
movies['crew']=movies['crew'].apply(fetch_director)

In [22]:
# Converting overview to list
movies['overview']= movies['overview'].apply(lambda x:x.split())

In [23]:
# Removing spaces between two words (eg. name and surname)
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [24]:
# A new column (tags)
movies['tags']= movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [25]:
new_df= movies[['movie_id','title','tags']]

In [26]:
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x)) # Converting list to single string

/tmp/ipykernel_3293/2151873370.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x)) # Converting list to single string


In [27]:
new_df['tags'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [28]:
new_df['tags']=new_df['tags'].apply(lambda x:x.lower()) # Convert to lowercase

/tmp/ipykernel_3293/1480731799.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x:x.lower()) # Convert to lowercase


In [29]:
# Final DataFrame after data pre-processing
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [30]:
from sklearn.feature_extraction.text import CountVectorizer
cv= CountVectorizer(max_features=5000,stop_words='english')

In [31]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [32]:
import nltk
from nltk.stem.porter import PorterStemmer
ps= PorterStemmer()

In [33]:
# Generalising similar words as one entity (same words won't be repeated)
def stem(text):
  y=[]
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)

In [34]:
# ps.stem('loved')
# ps.stem('loving')

In [35]:
new_df['tags']= new_df['tags'].apply(stem)

/tmp/ipykernel_3293/3021978705.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']= new_df['tags'].apply(stem)


In [36]:
from sklearn.metrics.pairwise import cosine_similarity
similarity= cosine_similarity(vectors)

In [37]:
# Similarity of second (index 1) with other movies
similarity[1]

array([0.08858079, 1.        , 0.06451613, ..., 0.02707652, 0.        ,
       0.        ])

In [38]:
# Sorted in descending order and kept index of comparator movie with it
# Printing top 5, most similar to least similar
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

[(539, np.float64(0.2537477434955704)),
 (1192, np.float64(0.25112360116696136)),
 (260, np.float64(0.2442250043490654)),
 (1214, np.float64(0.24260844762468367)),
 (507, np.float64(0.24165612634006073))]

In [39]:
def recommend(movie):
  movie_index= new_df[new_df['title']==movie].index[0] # Getting movie index
  distances= similarity[movie_index] # Getting similarity distances
  movie_list= sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

  for i in movie_list:
    print(new_df.iloc[i[0]].title) # Printing top 5 similar movie titles


In [40]:
recommend('Avatar') # Top 5 similar movies

Titan A.E.
Small Soldiers
Ender's Game
Aliens vs Predator: Requiem
Independence Day


In [41]:
recommend('Batman Begins')

The Dark Knight
The Dark Knight Rises
Batman
Batman
Batman & Robin


In [45]:
'''
import pickle

pickle.dump(new_df, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

movies = pickle.load(open('movies.pkl', 'rb'))
similarity = pickle.load(open('similarity.pkl', 'rb'))

'''

"\nimport pickle\n\npickle.dump(new_df, open('movies.pkl', 'wb'))\npickle.dump(similarity, open('similarity.pkl', 'wb'))\n\nmovies = pickle.load(open('movies.pkl', 'rb'))\nsimilarity = pickle.load(open('similarity.pkl', 'rb'))\n\n"